In [27]:
import pandas as pd
import re

In [28]:
df = pd.read_csv("../../data/raw/Laptop_Train_v2.csv")
df.head()

,id,Sentence,Aspect Term,polarity,from,to
0,2339,I charge it at night and skip taking the cord ...,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord ...,battery life,positive,74,86
2,1316,The tech guy then said the service center does...,service center,negative,27,41
3,1316,The tech guy then said the service center does...,"""sales"" team",negative,109,121
4,1316,The tech guy then said the service center does...,tech guy,neutral,4,12


In [29]:
# Missing values

df.columns
df.isnull().sum()


id             0
Sentence       0
Aspect Term    0
polarity       0
from           0
to             0
dtype: int64

In [30]:
# checks how many positive, negative and neutral reviews are there
# conflict -> both positive and negative reviews in a single review

df['polarity'].value_counts()

polarity
positive    987
negative    866
neutral     460
conflict     45
Name: count, dtype: int64

In [31]:
# removing conflict reviews due to low number of samples

df = df[df['polarity'] != 'conflict']
df['polarity'].value_counts()


polarity
positive    987
negative    866
neutral     460
Name: count, dtype: int64

In [32]:
df.head()


,id,Sentence,Aspect Term,polarity,from,to
0,2339,I charge it at night and skip taking the cord ...,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord ...,battery life,positive,74,86
2,1316,The tech guy then said the service center does...,service center,negative,27,41
3,1316,The tech guy then said the service center does...,"""sales"" team",negative,109,121
4,1316,The tech guy then said the service center does...,tech guy,neutral,4,12


In [33]:
# cleaning text

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['clean_sentence'] = df['Sentence'].apply(clean_text)

df[['Sentence', 'clean_sentence']].head()

,Sentence,clean_sentence
0,I charge it at night and skip taking the cord ...,i charge it at night and skip taking the cord ...
1,I charge it at night and skip taking the cord ...,i charge it at night and skip taking the cord ...
2,The tech guy then said the service center does...,the tech guy then said the service center does...
3,The tech guy then said the service center does...,the tech guy then said the service center does...
4,The tech guy then said the service center does...,the tech guy then said the service center does...


In [34]:
# extracting aspect context and replacing aspect term with [ASPECT] token to help model focus on the aspect


def extract_aspect_context(row, window=5):
    sentence = row['clean_sentence']
    aspect = row['Aspect Term'].lower()
    
    words = sentence.split()
    aspect_words = aspect.split()
    
    try:
        idx = words.index(aspect_words[0])
    except ValueError:
        return sentence
    
    left_context = words[max(0,idx-window) : idx]
    right_context = words[idx+len(aspect_words) : idx+len(aspect_words)+window]
    
    return " ".join(left_context + ["[ASPECT]"] + right_context)


In [35]:
# apply the function to extract aspect context

df['aspect_context'] = df.apply(extract_aspect_context, axis=1)
df[['clean_sentence', 'Aspect Term', 'aspect_context']].head()

,clean_sentence,Aspect Term,aspect_context
0,i charge it at night and skip taking the cord ...,cord,night and skip taking the [ASPECT] with me bec...
1,i charge it at night and skip taking the cord ...,battery life,me because of the good [ASPECT]
2,the tech guy then said the service center does...,service center,tech guy then said the [ASPECT] does not do 1-...
3,the tech guy then said the service center does...,"""sales"" team",direct my concern to the [ASPECT] which is the...
4,the tech guy then said the service center does...,tech guy,the [ASPECT] then said the service center


In [36]:
# encode labels

label_map = {

    'negative': 0,
    'neutral': 1,
    'positive': 2

}

df['label'] = df['polarity'].map(label_map)
df[['polarity', 'label']].head()


,polarity,label
0,neutral,1
1,positive,2
2,negative,0
3,negative,0
4,neutral,1


In [37]:
df['label'].value_counts()


label
2    987
0    866
1    460
Name: count, dtype: int64

In [ ]:
df[['aspect_context', 'label']].to_csv(
    "../../data/processed/laptop_train_processed.csv",
    index=False
)


# after this the only model inputs are 'aspect_context' and 'label'